In [2]:
!pip install -q gradio

import gradio as gr

head_html = r"""
<link href="https://fonts.googleapis.com/css2?family=Inconsolata:wght@400;600;700&display=swap" rel="stylesheet">
<style>
.fabric-app,
.fabric-app * {
  font-family: 'Inconsolata', ui-monospace, monospace;
}

.fabric-app {
  display: grid;
  grid-template-columns: 370px minmax(420px, 1fr);
  gap: 20px;
  width: 100%;
}

.fabric-card {
  background: var(--block-background-fill);
  border: 1px solid var(--border-color-primary);
  border-radius: 12px;
  padding: 16px;
}

.fabric-controls {
  max-height: 78vh;
  overflow-y: auto;
}

.fabric-section {
  border-top: 1px solid var(--border-color-primary);
  margin-top: 16px;
  padding-top: 12px;
}

.fabric-label {
  font-size: 13px;
  font-weight: 600;
  margin-top: 10px;
  display: block;
}

.fabric-select {
  width: 100%;
  margin-top: 6px;
  border-radius: 8px;
  border: 1px solid var(--border-color-primary);
  padding: 8px;
  background: var(--input-background-fill);
  color: var(--body-text-color);
}

.fabric-color {
  width: 100%;
  height: 38px;
  margin-top: 6px;
  border: 1px solid var(--border-color-primary);
  border-radius: 8px;
  background: transparent;
}

.fabric-range {
  width: 100%;
  margin-top: 6px;
}

.fabric-button {
  width: 100%;
  padding: 10px 14px;
  border-radius: 8px;
  border: none;
  margin-top: 12px;
  cursor: pointer;
  font-weight: 700;
  color: white;
  background: #2563eb;
}

.fabric-button.secondary {
  background: #16a34a;
}

.fabric-button.danger {
  background: #dc2626;
}

.fabric-button.small {
  padding: 6px 8px;
  font-size: 12px;
  margin-top: 6px;
}

.fabric-canvas {
  width: 100%;
  max-width: 900px;
  border-radius: 12px;
  border: 1px solid var(--border-color-primary);
  background: white;
  cursor: grab;
}

.fabric-canvas.dragging {
  cursor: grabbing;
}

.fabric-small {
  font-size: 12px;
  color: var(--body-text-color-subdued);
}

.stripe-item {
  border: 1px solid var(--border-color-primary);
  border-radius: 8px;
  padding: 8px;
  margin-top: 8px;
  font-size: 12px;
  background: var(--input-background-fill);
}

.stripe-title {
  font-weight: 700;
}

@media (max-width: 900px) {
  .fabric-app {
    grid-template-columns: 1fr;
  }
}
</style>

<script>
let customStripes = [];
let draggingStripeId = null;
let draggingOffset = 0;

// Fixed standard defaults.
// These are intentionally removed from the UI.
const DEFAULT_TEXTURE_AMOUNT = 0.85;
const DEFAULT_SOFTNESS = 0.10;
const DEFAULT_INTERSECTION_DARKNESS = 0.40;

function hexToRgb(hex) {
  hex = hex.replace("#", "");
  return {
    r: parseInt(hex.substring(0, 2), 16),
    g: parseInt(hex.substring(2, 4), 16),
    b: parseInt(hex.substring(4, 6), 16)
  };
}

function rgbToString(c) {
  return "rgb(" + Math.round(c.r) + "," + Math.round(c.g) + "," + Math.round(c.b) + ")";
}

function clamp(v) {
  return Math.max(0, Math.min(255, v));
}

function mix(c1, c2, amount) {
  return {
    r: c1.r * amount + c2.r * (1 - amount),
    g: c1.g * amount + c2.g * (1 - amount),
    b: c1.b * amount + c2.b * (1 - amount)
  };
}

function darken(c, amount) {
  return {
    r: clamp(c.r * (1 - amount)),
    g: clamp(c.g * (1 - amount)),
    b: clamp(c.b * (1 - amount))
  };
}

function lighten(c, amount) {
  return {
    r: clamp(c.r + (255 - c.r) * amount),
    g: clamp(c.g + (255 - c.g) * amount),
    b: clamp(c.b + (255 - c.b) * amount)
  };
}

function adjustColor(c, amount) {
  return {
    r: clamp(c.r + amount),
    g: clamp(c.g + amount),
    b: clamp(c.b + amount)
  };
}

function rectIntersection(a, b) {
  let x1 = Math.max(a.x, b.x);
  let y1 = Math.max(a.y, b.y);
  let x2 = Math.min(a.x + a.w, b.x + b.w);
  let y2 = Math.min(a.y + a.h, b.y + b.h);

  if (x2 <= x1 || y2 <= y1) return null;

  return {
    x: x1,
    y: y1,
    w: x2 - x1,
    h: y2 - y1
  };
}

function addFiberNoise(tctx, tileWidth, tileHeight, cw, ch, textureAmount) {
  const slubCount = Math.floor(20 + textureAmount * 90);

  for (let i = 0; i < slubCount; i++) {
    const x = Math.random() * tileWidth;
    const y = Math.random() * tileHeight;

    tctx.globalAlpha = Math.random() * 0.14;
    tctx.fillStyle = Math.random() > 0.45 ? "white" : "black";

    const dotSize = Math.random() > 0.85 ? 2 : 1;
    tctx.fillRect(x, y, dotSize, dotSize);
  }

  const fiberCount = Math.floor(20 + textureAmount * 70);

  for (let i = 0; i < fiberCount; i++) {
    const horizontal = Math.random() > 0.5;

    tctx.globalAlpha = Math.random() * 0.07;
    tctx.strokeStyle = Math.random() > 0.5 ? "white" : "black";
    tctx.lineWidth = 1;

    tctx.beginPath();

    if (horizontal) {
      const y = Math.random() * tileHeight;
      const x1 = Math.random() * tileWidth;
      const x2 = x1 + Math.random() * cw * 4;
      tctx.moveTo(x1, y);
      tctx.lineTo(x2, y + (Math.random() - 0.5) * 1.5);
    } else {
      const x = Math.random() * tileWidth;
      const y1 = Math.random() * tileHeight;
      const y2 = y1 + Math.random() * ch * 4;
      tctx.moveTo(x, y1);
      tctx.lineTo(x + (Math.random() - 0.5) * 1.5, y2);
    }

    tctx.stroke();
  }

  tctx.globalAlpha = 1.0;
}

function createPlainWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount) {
  const cw = Math.max(2, warpThickness);
  const ch = Math.max(2, weftThickness);

  const repeatCols = 18;
  const repeatRows = 18;

  const tileWidth = cw * repeatCols;
  const tileHeight = ch * repeatRows;

  const tile = document.createElement("canvas");
  tile.width = tileWidth;
  tile.height = tileHeight;

  const tctx = tile.getContext("2d");

  tctx.fillStyle = rgbToString(mix(warpColor, weftColor, 0.5));
  tctx.fillRect(0, 0, tileWidth, tileHeight);

  let colVariation = [];
  let rowVariation = [];

  for (let i = 0; i < repeatCols; i++) {
    colVariation.push((Math.random() - 0.5) * 18 * textureAmount);
  }

  for (let i = 0; i < repeatRows; i++) {
    rowVariation.push((Math.random() - 0.5) * 18 * textureAmount);
  }

  for (let row = 0; row < repeatRows; row++) {
    for (let col = 0; col < repeatCols; col++) {
      const x = col * cw;
      const y = row * ch;

      const warpOnTop = (row + col) % 2 === 0;

      let base = warpOnTop ? warpColor : weftColor;
      let under = warpOnTop ? weftColor : warpColor;

      let woven = mix(base, under, 0.86);

      let variation = warpOnTop ? colVariation[col] : rowVariation[row];
      woven = adjustColor(woven, variation);

      woven.r = clamp(woven.r + (Math.random() - 0.5) * 4 * textureAmount);
      woven.g = clamp(woven.g + (Math.random() - 0.5) * 3 * textureAmount);
      woven.b = clamp(woven.b + (Math.random() - 0.5) * 4 * textureAmount);

      tctx.fillStyle = rgbToString(woven);
      tctx.fillRect(x, y, cw, ch);

      tctx.globalAlpha = 0.10 + textureAmount * 0.10;
      tctx.fillStyle = "white";

      if (warpOnTop) {
        tctx.fillRect(x + Math.floor(cw * 0.35), y, 1, ch);
      } else {
        tctx.fillRect(x, y + Math.floor(ch * 0.35), cw, 1);
      }

      tctx.globalAlpha = 0.08 + textureAmount * 0.10;
      tctx.fillStyle = "black";

      if (warpOnTop) {
        tctx.fillRect(x + cw - 1, y, 1, ch);
      } else {
        tctx.fillRect(x, y + ch - 1, cw, 1);
      }

      tctx.globalAlpha = 1.0;
    }
  }

  tctx.globalAlpha = 0.06 + textureAmount * 0.09;
  tctx.strokeStyle = "black";
  tctx.lineWidth = 1;

  for (let x = 0; x <= tileWidth; x += cw) {
    tctx.beginPath();
    tctx.moveTo(x + 0.5, 0);
    tctx.lineTo(x + 0.5, tileHeight);
    tctx.stroke();
  }

  for (let y = 0; y <= tileHeight; y += ch) {
    tctx.beginPath();
    tctx.moveTo(0, y + 0.5);
    tctx.lineTo(tileWidth, y + 0.5);
    tctx.stroke();
  }

  addFiberNoise(tctx, tileWidth, tileHeight, cw, ch, textureAmount);

  tctx.globalAlpha = 1.0;

  return ctx.createPattern(tile, "repeat");
}

function createWaffleWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount) {
  const cw = Math.max(2, warpThickness);
  const ch = Math.max(2, weftThickness);

  const waffleDepth = parseInt(document.getElementById("waffleDepth").value) / 100;
  const waffleCellScale = parseInt(document.getElementById("waffleCellScale").value);

  const unitW = Math.max(22, cw * waffleCellScale);
  const unitH = Math.max(22, ch * waffleCellScale);

  const repeatCols = 7;
  const repeatRows = 7;

  const tileWidth = unitW * repeatCols;
  const tileHeight = unitH * repeatRows;

  const ridgeW = Math.max(4, Math.floor(unitW * 0.24));
  const ridgeH = Math.max(4, Math.floor(unitH * 0.24));

  const tile = document.createElement("canvas");
  tile.width = tileWidth;
  tile.height = tileHeight;

  const tctx = tile.getContext("2d");

  const baseColor = mix(warpColor, weftColor, 0.5);
  const raisedLight = lighten(baseColor, 0.16 + waffleDepth * 0.20);
  const raisedMid = lighten(baseColor, 0.05 + waffleDepth * 0.08);
  const recessedMid = darken(baseColor, 0.06 + waffleDepth * 0.15);
  const recessedDark = darken(baseColor, 0.12 + waffleDepth * 0.22);
  const deepShadow = darken(baseColor, 0.20 + waffleDepth * 0.24);

  tctx.fillStyle = rgbToString(baseColor);
  tctx.fillRect(0, 0, tileWidth, tileHeight);

  for (let row = 0; row < repeatRows; row++) {
    for (let col = 0; col < repeatCols; col++) {
      const x = col * unitW;
      const y = row * unitH;

      const variation = (Math.random() - 0.5) * 14 * textureAmount;

      const localRaisedLight = adjustColor(raisedLight, variation);
      const localRaisedMid = adjustColor(raisedMid, variation * 0.65);
      const localRecessedMid = adjustColor(recessedMid, variation * 0.45);
      const localRecessedDark = adjustColor(recessedDark, variation * 0.35);
      const localDeepShadow = adjustColor(deepShadow, variation * 0.30);

      const centerX = x + ridgeW;
      const centerY = y + ridgeH;
      const centerW = unitW - ridgeW * 2;
      const centerH = unitH - ridgeH * 2;

      tctx.fillStyle = rgbToString(localRaisedMid);
      tctx.fillRect(x, y, unitW, unitH);

      const pocketGrad = tctx.createLinearGradient(centerX, centerY, centerX + centerW, centerY + centerH);
      pocketGrad.addColorStop(0.00, rgbToString(localRecessedDark));
      pocketGrad.addColorStop(0.35, rgbToString(localRecessedMid));
      pocketGrad.addColorStop(0.70, rgbToString(darken(localRecessedMid, 0.06)));
      pocketGrad.addColorStop(1.00, rgbToString(localDeepShadow));

      tctx.fillStyle = pocketGrad;
      tctx.fillRect(centerX, centerY, centerW, centerH);

      const topGrad = tctx.createLinearGradient(x, y, x, y + ridgeH);
      topGrad.addColorStop(0.00, rgbToString(localRaisedLight));
      topGrad.addColorStop(0.45, rgbToString(localRaisedMid));
      topGrad.addColorStop(1.00, rgbToString(darken(localRaisedMid, 0.08)));

      tctx.fillStyle = topGrad;
      tctx.fillRect(x, y, unitW, ridgeH);

      const leftGrad = tctx.createLinearGradient(x, y, x + ridgeW, y);
      leftGrad.addColorStop(0.00, rgbToString(localRaisedLight));
      leftGrad.addColorStop(0.50, rgbToString(localRaisedMid));
      leftGrad.addColorStop(1.00, rgbToString(darken(localRaisedMid, 0.08)));

      tctx.fillStyle = leftGrad;
      tctx.fillRect(x, y, ridgeW, unitH);

      const bottomGrad = tctx.createLinearGradient(x, y + unitH - ridgeH, x, y + unitH);
      bottomGrad.addColorStop(0.00, rgbToString(darken(localRaisedMid, 0.06)));
      bottomGrad.addColorStop(1.00, rgbToString(localDeepShadow));

      tctx.fillStyle = bottomGrad;
      tctx.fillRect(x, y + unitH - ridgeH, unitW, ridgeH);

      const rightGrad = tctx.createLinearGradient(x + unitW - ridgeW, y, x + unitW, y);
      rightGrad.addColorStop(0.00, rgbToString(darken(localRaisedMid, 0.05)));
      rightGrad.addColorStop(1.00, rgbToString(localDeepShadow));

      tctx.fillStyle = rightGrad;
      tctx.fillRect(x + unitW - ridgeW, y, ridgeW, unitH);

      tctx.globalAlpha = 0.16 + waffleDepth * 0.18;
      tctx.fillStyle = "black";
      tctx.fillRect(centerX, centerY, centerW, 1);
      tctx.fillRect(centerX, centerY, 1, centerH);

      tctx.globalAlpha = 0.16 + textureAmount * 0.10;
      tctx.fillStyle = "white";
      tctx.fillRect(x + Math.floor(ridgeW * 0.38), y, 1, unitH);
      tctx.fillRect(x, y + Math.floor(ridgeH * 0.38), unitW, 1);

      tctx.globalAlpha = 1.0;
    }
  }

  const fuzzCount = Math.floor(60 + textureAmount * 160);

  for (let i = 0; i < fuzzCount; i++) {
    const horizontal = Math.random() > 0.42;

    const fx = Math.random() * tileWidth;
    const fy = Math.random() * tileHeight;
    const len = 2 + Math.random() * Math.max(cw, ch) * 3;

    tctx.globalAlpha = Math.random() * 0.13;
    tctx.strokeStyle = Math.random() > 0.25 ? "rgba(255,255,255,0.95)" : "rgba(95,85,70,0.65)";
    tctx.lineWidth = Math.random() > 0.85 ? 2 : 1;

    tctx.beginPath();

    if (horizontal) {
      tctx.moveTo(fx, fy);
      tctx.lineTo(fx + len, fy + (Math.random() - 0.5) * 2);
    } else {
      tctx.moveTo(fx, fy);
      tctx.lineTo(fx + (Math.random() - 0.5) * 2, fy + len);
    }

    tctx.stroke();
  }

  tctx.globalAlpha = 0.10 + waffleDepth * 0.12;
  tctx.strokeStyle = "rgba(60,55,45,0.55)";
  tctx.lineWidth = 1;

  for (let x = 0; x <= tileWidth; x += unitW) {
    tctx.beginPath();
    tctx.moveTo(x + 0.5, 0);
    tctx.lineTo(x + 0.5, tileHeight);
    tctx.stroke();
  }

  for (let y = 0; y <= tileHeight; y += unitH) {
    tctx.beginPath();
    tctx.moveTo(0, y + 0.5);
    tctx.lineTo(tileWidth, y + 0.5);
    tctx.stroke();
  }

  tctx.globalAlpha = 1.0;

  return ctx.createPattern(tile, "repeat");
}

function createLooseWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount) {
  const cw = Math.max(1, warpThickness);
  const ch = Math.max(1, weftThickness);

  const openness = parseInt(document.getElementById("looseOpenness").value) / 100;
  const looseIrregularity = parseInt(document.getElementById("looseIrregularity").value) / 100;
  const looseThreadOpacity = parseInt(document.getElementById("looseThreadOpacity").value) / 100;

  const spacingX = Math.max(8, cw * (3.5 + openness * 9.5));
  const spacingY = Math.max(8, ch * (3.5 + openness * 9.5));

  const threadW = Math.max(1.1, cw * (0.65 - openness * 0.28));
  const threadH = Math.max(1.1, ch * (0.65 - openness * 0.28));

  const repeatCols = 24;
  const repeatRows = 24;

  const tileWidth = Math.ceil(spacingX * repeatCols);
  const tileHeight = Math.ceil(spacingY * repeatRows);

  const tile = document.createElement("canvas");
  tile.width = tileWidth;
  tile.height = tileHeight;

  const tctx = tile.getContext("2d");

  const baseColor = lighten(mix(warpColor, weftColor, 0.5), 0.30);
  const gapColor = lighten(baseColor, 0.08);
  const gapShadow = darken(baseColor, 0.15);

  const warpMain = lighten(warpColor, 0.04);
  const weftMain = lighten(weftColor, 0.04);

  const warpShadow = darken(warpColor, 0.25);
  const weftShadow = darken(weftColor, 0.25);

  const warpHighlight = lighten(warpColor, 0.35);
  const weftHighlight = lighten(weftColor, 0.35);

  tctx.fillStyle = rgbToString(gapColor);
  tctx.fillRect(0, 0, tileWidth, tileHeight);

  tctx.globalAlpha = 0.07 + textureAmount * 0.06;
  tctx.fillStyle = rgbToString(gapShadow);

  for (let y = 0; y < tileHeight; y += spacingY) {
    for (let x = 0; x < tileWidth; x += spacingX) {
      const jx = (Math.random() - 0.5) * spacingX * looseIrregularity * 0.35;
      const jy = (Math.random() - 0.5) * spacingY * looseIrregularity * 0.35;

      tctx.fillRect(
        x + threadW * 1.4 + jx,
        y + threadH * 1.4 + jy,
        Math.max(1, spacingX - threadW * 3.0),
        Math.max(1, spacingY - threadH * 3.0)
      );
    }
  }

  tctx.globalAlpha = 1.0;

  let warpXs = [];
  let weftYs = [];

  for (let i = -1; i <= repeatCols + 1; i++) {
    const jitter = (Math.random() - 0.5) * spacingX * looseIrregularity * 0.42;
    warpXs.push(i * spacingX + jitter);
  }

  for (let i = -1; i <= repeatRows + 1; i++) {
    const jitter = (Math.random() - 0.5) * spacingY * looseIrregularity * 0.42;
    weftYs.push(i * spacingY + jitter);
  }

  function drawVerticalThread(x, colorMain, colorShadow, colorHighlight, width, alpha) {
    const wave = spacingX * looseIrregularity * 0.10;
    const segment = Math.max(18, spacingY * 1.6);

    function drawPath(offsetX, strokeColor, strokeWidth, opacity) {
      tctx.globalAlpha = opacity;
      tctx.strokeStyle = strokeColor;
      tctx.lineWidth = strokeWidth;
      tctx.lineCap = "round";

      tctx.beginPath();
      tctx.moveTo(x + offsetX + (Math.random() - 0.5) * wave, -10);

      for (let y = 0; y <= tileHeight + segment; y += segment) {
        const cx = x + offsetX + (Math.random() - 0.5) * wave;
        const cy = y + segment * 0.5;
        const ex = x + offsetX + (Math.random() - 0.5) * wave;
        const ey = y + segment;

        tctx.quadraticCurveTo(cx, cy, ex, ey);
      }

      tctx.stroke();
    }

    drawPath(0.8, rgbToString(colorShadow), width * 1.55, alpha * 0.34);
    drawPath(0, rgbToString(colorMain), width, alpha);
    drawPath(-width * 0.18, rgbToString(colorHighlight), Math.max(0.55, width * 0.28), alpha * 0.42);
    tctx.globalAlpha = 1.0;
  }

  function drawHorizontalThread(y, colorMain, colorShadow, colorHighlight, width, alpha) {
    const wave = spacingY * looseIrregularity * 0.10;
    const segment = Math.max(18, spacingX * 1.6);

    function drawPath(offsetY, strokeColor, strokeWidth, opacity) {
      tctx.globalAlpha = opacity;
      tctx.strokeStyle = strokeColor;
      tctx.lineWidth = strokeWidth;
      tctx.lineCap = "round";

      tctx.beginPath();
      tctx.moveTo(-10, y + offsetY + (Math.random() - 0.5) * wave);

      for (let x = 0; x <= tileWidth + segment; x += segment) {
        const cx = x + segment * 0.5;
        const cy = y + offsetY + (Math.random() - 0.5) * wave;
        const ex = x + segment;
        const ey = y + offsetY + (Math.random() - 0.5) * wave;

        tctx.quadraticCurveTo(cx, cy, ex, ey);
      }

      tctx.stroke();
    }

    drawPath(0.8, rgbToString(colorShadow), width * 1.55, alpha * 0.34);
    drawPath(0, rgbToString(colorMain), width, alpha);
    drawPath(-width * 0.18, rgbToString(colorHighlight), Math.max(0.55, width * 0.28), alpha * 0.42);
    tctx.globalAlpha = 1.0;
  }

  for (let i = 0; i < warpXs.length; i++) {
    const alpha = looseThreadOpacity * (0.55 + Math.random() * 0.28);
    drawVerticalThread(
      warpXs[i],
      adjustColor(warpMain, (Math.random() - 0.5) * 18 * textureAmount),
      warpShadow,
      warpHighlight,
      threadW,
      alpha
    );
  }

  for (let j = 0; j < weftYs.length; j++) {
    const alpha = looseThreadOpacity * (0.55 + Math.random() * 0.28);
    drawHorizontalThread(
      weftYs[j],
      adjustColor(weftMain, (Math.random() - 0.5) * 18 * textureAmount),
      weftShadow,
      weftHighlight,
      threadH,
      alpha
    );
  }

  for (let i = 0; i < warpXs.length; i++) {
    for (let j = 0; j < weftYs.length; j++) {
      const x = warpXs[i];
      const y = weftYs[j];

      const warpOnTop = (i + j) % 2 === 0;

      if (warpOnTop) {
        tctx.globalAlpha = looseThreadOpacity * 0.80;
        tctx.strokeStyle = rgbToString(lighten(warpMain, 0.12));
        tctx.lineWidth = threadW * 1.08;
        tctx.lineCap = "round";

        tctx.beginPath();
        tctx.moveTo(x, y - spacingY * 0.28);
        tctx.lineTo(x + (Math.random() - 0.5) * 1.5, y + spacingY * 0.28);
        tctx.stroke();
      } else {
        tctx.globalAlpha = looseThreadOpacity * 0.80;
        tctx.strokeStyle = rgbToString(lighten(weftMain, 0.12));
        tctx.lineWidth = threadH * 1.08;
        tctx.lineCap = "round";

        tctx.beginPath();
        tctx.moveTo(x - spacingX * 0.28, y);
        tctx.lineTo(x + spacingX * 0.28, y + (Math.random() - 0.5) * 1.5);
        tctx.stroke();
      }
    }
  }

  tctx.globalAlpha = 0.08 + openness * 0.10;
  tctx.fillStyle = "rgba(255,255,255,0.85)";

  for (let y = 0; y < tileHeight; y += spacingY) {
    for (let x = 0; x < tileWidth; x += spacingX) {
      tctx.fillRect(
        x + threadW * 1.8,
        y + threadH * 1.8,
        Math.max(1, spacingX - threadW * 4.0),
        Math.max(1, spacingY - threadH * 4.0)
      );
    }
  }

  const strayCount = Math.floor(80 + textureAmount * 220);

  for (let i = 0; i < strayCount; i++) {
    const horizontal = Math.random() > 0.48;
    const fx = Math.random() * tileWidth;
    const fy = Math.random() * tileHeight;
    const len = 6 + Math.random() * Math.max(spacingX, spacingY) * 2.2;

    tctx.globalAlpha = Math.random() * (0.08 + textureAmount * 0.13);
    tctx.strokeStyle = Math.random() > 0.22 ? "rgba(255,255,255,0.95)" : "rgba(120,112,96,0.50)";
    tctx.lineWidth = Math.random() > 0.90 ? 2 : 1;
    tctx.lineCap = "round";

    tctx.beginPath();

    if (horizontal) {
      tctx.moveTo(fx, fy);
      tctx.lineTo(fx + len, fy + (Math.random() - 0.5) * 4);
    } else {
      tctx.moveTo(fx, fy);
      tctx.lineTo(fx + (Math.random() - 0.5) * 4, fy + len);
    }

    tctx.stroke();
  }

  tctx.globalAlpha = 1.0;

  return ctx.createPattern(tile, "repeat");
}

function createSelectedWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount) {
  const patternType = document.getElementById("patternType").value;

  if (patternType === "waffle") {
    return createWaffleWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount);
  }

  if (patternType === "loose") {
    return createLooseWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount);
  }

  return createPlainWeavePattern(ctx, warpColor, weftColor, warpThickness, weftThickness, textureAmount);
}

function fillPatternRect(ctx, pattern, rect) {
  ctx.save();
  ctx.fillStyle = pattern;
  ctx.fillRect(rect.x, rect.y, rect.w, rect.h);
  ctx.restore();
}

function applyGlobalSoftness(ctx, width, height, softness) {
  if (softness <= 0) return;

  const temp = document.createElement("canvas");
  temp.width = width;
  temp.height = height;

  const tctx = temp.getContext("2d");
  tctx.drawImage(ctx.canvas, 0, 0);

  ctx.clearRect(0, 0, width, height);
  ctx.filter = "blur(" + Math.min(0.65, softness) + "px)";
  ctx.drawImage(temp, 0, 0);
  ctx.filter = "none";
}

function drawRulers(ctx, width, height) {
  const showRulers = document.getElementById("showRulers").checked;
  if (!showRulers) return;

  const rulerSize = 42;
  const pixelsPerCm = parseInt(document.getElementById("pixelsPerCm").value);

  const minorTick = pixelsPerCm / 10;
  const oneCm = pixelsPerCm;

  ctx.save();

  ctx.fillStyle = "rgba(250, 250, 245, 0.90)";
  ctx.fillRect(0, 0, width, rulerSize);
  ctx.fillRect(0, 0, rulerSize, height);

  ctx.fillStyle = "rgba(235, 235, 225, 0.96)";
  ctx.fillRect(0, 0, rulerSize, rulerSize);

  ctx.strokeStyle = "rgba(20, 20, 20, 0.85)";
  ctx.fillStyle = "rgba(20, 20, 20, 0.90)";
  ctx.lineWidth = 1;
  ctx.font = "10px Inconsolata";
  ctx.textBaseline = "top";

  for (let x = 0; x <= width; x += minorTick) {
    const tickIndex = Math.round(x / minorTick);
    const isCm = tickIndex % 10 === 0;
    const isHalfCm = tickIndex % 5 === 0;

    let tickLength = rulerSize * 0.32;

    if (isHalfCm) tickLength = rulerSize * 0.50;
    if (isCm) tickLength = rulerSize * 0.78;

    ctx.beginPath();
    ctx.moveTo(x + 0.5, 0);
    ctx.lineTo(x + 0.5, tickLength);
    ctx.stroke();

    if (isCm) {
      const cmNumber = Math.round(x / oneCm);

      if (cmNumber === 0) {
        ctx.fillText("0", 4, rulerSize * 0.56);
      } else {
        ctx.fillText(String(cmNumber), x + 3, rulerSize * 0.56);
      }
    }
  }

  for (let y = 0; y <= height; y += minorTick) {
    const tickIndex = Math.round(y / minorTick);
    const isCm = tickIndex % 10 === 0;
    const isHalfCm = tickIndex % 5 === 0;

    let tickLength = rulerSize * 0.32;

    if (isHalfCm) tickLength = rulerSize * 0.50;
    if (isCm) tickLength = rulerSize * 0.78;

    ctx.beginPath();
    ctx.moveTo(0, y + 0.5);
    ctx.lineTo(tickLength, y + 0.5);
    ctx.stroke();

    if (isCm) {
      const cmNumber = Math.round(y / oneCm);

      if (cmNumber === 0) {
        ctx.fillText("0", 6, 6);
      } else {
        ctx.save();
        ctx.translate(rulerSize * 0.58, y + 3);
        ctx.rotate(-Math.PI / 2);
        ctx.fillText(String(cmNumber), 0, 0);
        ctx.restore();
      }
    }
  }

  ctx.strokeStyle = "rgba(20, 20, 20, 0.65)";
  ctx.beginPath();
  ctx.moveTo(0, rulerSize + 0.5);
  ctx.lineTo(width, rulerSize + 0.5);
  ctx.moveTo(rulerSize + 0.5, 0);
  ctx.lineTo(rulerSize + 0.5, height);
  ctx.stroke();

  ctx.fillStyle = "rgba(20, 20, 20, 0.85)";
  ctx.font = "9px Inconsolata";
  ctx.fillText("cm", rulerSize - 18, rulerSize - 14);

  ctx.restore();
}

function getCanvasMousePosition(evt) {
  const canvas = document.getElementById("fabricCanvas");
  const rect = canvas.getBoundingClientRect();

  const scaleX = canvas.width / rect.width;
  const scaleY = canvas.height / rect.height;

  return {
    x: (evt.clientX - rect.left) * scaleX,
    y: (evt.clientY - rect.top) * scaleY
  };
}

function addVerticalStripe() {
  const size = parseInt(document.getElementById("size").value);

  customStripes.push({
    id: Date.now() + Math.random(),
    type: "vertical",
    position: Math.floor(size * 0.35),
    width: parseInt(document.getElementById("newStripeWidth").value),
    warpColor: document.getElementById("newStripeWarp").value,
    weftColor: document.getElementById("newStripeWeft").value
  });

  updateStripeList();
  renderFabricInstant();
}

function addHorizontalStripe() {
  const size = parseInt(document.getElementById("size").value);

  customStripes.push({
    id: Date.now() + Math.random(),
    type: "horizontal",
    position: Math.floor(size * 0.35),
    width: parseInt(document.getElementById("newStripeWidth").value),
    warpColor: document.getElementById("newStripeWarp").value,
    weftColor: document.getElementById("newStripeWeft").value
  });

  updateStripeList();
  renderFabricInstant();
}

function removeStripe(id) {
  customStripes = customStripes.filter(s => String(s.id) !== String(id));
  updateStripeList();
  renderFabricInstant();
}

function updateStripeList() {
  const list = document.getElementById("stripeList");
  if (!list) return;

  if (customStripes.length === 0) {
    list.innerHTML = "<p class='fabric-small'>No custom stripes yet.</p>";
    return;
  }

  let html = "";

  customStripes.forEach((s, index) => {
    html += `
      <div class="stripe-item">
        <div class="stripe-title">${index + 1}. ${s.type} stripe</div>
        Position: ${Math.round(s.position)} px<br>
        Width: ${Math.round(s.width)} px<br>
        Warp: ${s.warpColor} | Weft: ${s.weftColor}
        <button class="fabric-button danger small" onclick="removeStripe('${s.id}')">Remove</button>
      </div>
    `;
  });

  list.innerHTML = html;
}

function findStripeAtPoint(x, y) {
  for (let i = customStripes.length - 1; i >= 0; i--) {
    const s = customStripes[i];

    if (s.type === "vertical") {
      if (x >= s.position && x <= s.position + s.width) {
        return {
          stripe: s,
          offset: x - s.position
        };
      }
    }

    if (s.type === "horizontal") {
      if (y >= s.position && y <= s.position + s.width) {
        return {
          stripe: s,
          offset: y - s.position
        };
      }
    }
  }

  return null;
}

function setupCanvasDrag() {
  const canvas = document.getElementById("fabricCanvas");
  if (!canvas) return;

  canvas.addEventListener("mousedown", function(evt) {
    const pos = getCanvasMousePosition(evt);
    const hit = findStripeAtPoint(pos.x, pos.y);

    if (hit) {
      draggingStripeId = hit.stripe.id;
      draggingOffset = hit.offset;
      canvas.classList.add("dragging");
    }
  });

  window.addEventListener("mousemove", function(evt) {
    if (draggingStripeId === null) return;

    const canvas = document.getElementById("fabricCanvas");
    const pos = getCanvasMousePosition(evt);

    const stripe = customStripes.find(s => s.id === draggingStripeId);
    if (!stripe) return;

    if (stripe.type === "vertical") {
      stripe.position = Math.max(0, Math.min(canvas.width - stripe.width, pos.x - draggingOffset));
    } else {
      stripe.position = Math.max(0, Math.min(canvas.height - stripe.width, pos.y - draggingOffset));
    }

    updateStripeList();
    renderFabricInstant();
  });

  window.addEventListener("mouseup", function() {
    draggingStripeId = null;
    draggingOffset = 0;

    const canvas = document.getElementById("fabricCanvas");
    if (canvas) canvas.classList.remove("dragging");
  });
}

function renderFabricInstant() {
  const canvas = document.getElementById("fabricCanvas");
  if (!canvas) return;

  const ctx = canvas.getContext("2d");

  const size = parseInt(document.getElementById("size").value);
  canvas.width = size;
  canvas.height = size;

  const width = size;
  const height = size;

  const warpThickness = parseInt(document.getElementById("warpThickness").value);
  const weftThickness = parseInt(document.getElementById("weftThickness").value);

  const textureAmount = DEFAULT_TEXTURE_AMOUNT;
  const softness = DEFAULT_SOFTNESS;
  const intersectionDarkness = DEFAULT_INTERSECTION_DARKNESS;

  const bodyV = hexToRgb(document.getElementById("bodyV").value);
  const bodyH = hexToRgb(document.getElementById("bodyH").value);

  const borderV = hexToRgb(document.getElementById("borderV").value);
  const borderH = hexToRgb(document.getElementById("borderH").value);

  ctx.clearRect(0, 0, width, height);

  let verticalBands = [];
  let horizontalBands = [];

  const bodyPattern = createSelectedWeavePattern(
    ctx,
    bodyV,
    bodyH,
    warpThickness,
    weftThickness,
    textureAmount
  );

  fillPatternRect(ctx, bodyPattern, {x: 0, y: 0, w: width, h: height});

  if (document.getElementById("enableBorders").checked) {
    const top = parseInt(document.getElementById("topBorder").value);
    const bottom = parseInt(document.getElementById("bottomBorder").value);
    const left = parseInt(document.getElementById("leftBorder").value);
    const right = parseInt(document.getElementById("rightBorder").value);

    const verticalBorderPattern = createSelectedWeavePattern(
      ctx,
      borderV,
      bodyH,
      warpThickness,
      weftThickness,
      textureAmount
    );

    const horizontalBorderPattern = createSelectedWeavePattern(
      ctx,
      bodyV,
      borderH,
      warpThickness,
      weftThickness,
      textureAmount
    );

    if (left > 0) {
      const rect = {x: 0, y: 0, w: left, h: height};
      fillPatternRect(ctx, verticalBorderPattern, rect);
      verticalBands.push({rect: rect, vertical: borderV, horizontal: bodyH});
    }

    if (right > 0) {
      const rect = {x: width - right, y: 0, w: right, h: height};
      fillPatternRect(ctx, verticalBorderPattern, rect);
      verticalBands.push({rect: rect, vertical: borderV, horizontal: bodyH});
    }

    if (top > 0) {
      const rect = {x: 0, y: 0, w: width, h: top};
      fillPatternRect(ctx, horizontalBorderPattern, rect);
      horizontalBands.push({rect: rect, vertical: bodyV, horizontal: borderH});
    }

    if (bottom > 0) {
      const rect = {x: 0, y: height - bottom, w: width, h: bottom};
      fillPatternRect(ctx, horizontalBorderPattern, rect);
      horizontalBands.push({rect: rect, vertical: bodyV, horizontal: borderH});
    }
  }

  customStripes.forEach(s => {
    const warpColor = hexToRgb(s.warpColor);
    const weftColor = hexToRgb(s.weftColor);

    const stripePattern = createSelectedWeavePattern(
      ctx,
      warpColor,
      weftColor,
      warpThickness,
      weftThickness,
      textureAmount
    );

    if (s.type === "vertical") {
      const rect = {
        x: s.position,
        y: 0,
        w: s.width,
        h: height
      };

      fillPatternRect(ctx, stripePattern, rect);
      verticalBands.push({rect: rect, vertical: warpColor, horizontal: weftColor});
    }

    if (s.type === "horizontal") {
      const rect = {
        x: 0,
        y: s.position,
        w: width,
        h: s.width
      };

      fillPatternRect(ctx, stripePattern, rect);
      horizontalBands.push({rect: rect, vertical: warpColor, horizontal: weftColor});
    }
  });

  for (const v of verticalBands) {
    for (const h of horizontalBands) {
      const inter = rectIntersection(v.rect, h.rect);

      if (inter) {
        const interV = darken(v.vertical, intersectionDarkness * 0.16);
        const interH = darken(h.horizontal, intersectionDarkness * 0.16);

        const interPattern = createSelectedWeavePattern(
          ctx,
          interV,
          interH,
          warpThickness,
          weftThickness,
          textureAmount
        );

        fillPatternRect(ctx, interPattern, inter);
      }
    }
  }

  applyGlobalSoftness(ctx, width, height, softness * 0.7);

  drawRulers(ctx, width, height);
}

function downloadFabricPNG() {
  const canvas = document.getElementById("fabricCanvas");
  const patternType = document.getElementById("patternType").value;
  const link = document.createElement("a");
  link.download = "custom_" + patternType + "_woven_fabric.png";
  link.href = canvas.toDataURL("image/png");
  link.click();
}

function initFabricUI() {
  const canvas = document.getElementById("fabricCanvas");

  if (!canvas) {
    setTimeout(initFabricUI, 500);
    return;
  }

  document.querySelectorAll(".fabric-app input, .fabric-app select").forEach(el => {
    el.addEventListener("input", () => {
      clearTimeout(window.fabricRenderTimer);
      window.fabricRenderTimer = setTimeout(renderFabricInstant, 60);
    });
  });

  setupCanvasDrag();
  updateStripeList();
  renderFabricInstant();
}

setTimeout(initFabricUI, 1000);
</script>
"""

body_html = r"""
<div class="fabric-app">
  <div class="fabric-card fabric-controls">
    <h2>Fabric Controls</h2>
    <p class="fabric-small">
      Texture amount, softness, and intersection darkness are now fixed to standard defaults.
    </p>

    <div class="fabric-section">
      <h3>Pattern Type</h3>

      <label class="fabric-label">Choose weave pattern</label>
      <select class="fabric-select" id="patternType">
        <option value="plain">Plain weave</option>
        <option value="waffle">Waffle weave</option>
        <option value="loose" selected>Loose weave / gauze weave</option>
      </select>
    </div>

    <div class="fabric-section">
      <h3>Body</h3>

      <label class="fabric-label">Body vertical / warp color</label>
      <input class="fabric-color" id="bodyV" type="color" value="#e8e4d8">

      <label class="fabric-label">Body horizontal / weft color</label>
      <input class="fabric-color" id="bodyH" type="color" value="#f7f5ee">
    </div>

    <div class="fabric-section">
      <h3>Borders</h3>

      <label class="fabric-label">
        <input id="enableBorders" type="checkbox">
        Enable borders
      </label>

      <label class="fabric-label">Border vertical / warp color</label>
      <input class="fabric-color" id="borderV" type="color" value="#c8322d">

      <label class="fabric-label">Border horizontal / weft color</label>
      <input class="fabric-color" id="borderH" type="color" value="#c8322d">

      <label class="fabric-label">Top border</label>
      <input class="fabric-range" id="topBorder" type="range" min="0" max="300" value="0">

      <label class="fabric-label">Bottom border</label>
      <input class="fabric-range" id="bottomBorder" type="range" min="0" max="300" value="0">

      <label class="fabric-label">Left border</label>
      <input class="fabric-range" id="leftBorder" type="range" min="0" max="300" value="0">

      <label class="fabric-label">Right border</label>
      <input class="fabric-range" id="rightBorder" type="range" min="0" max="300" value="0">
    </div>

    <div class="fabric-section">
      <h3>Add Custom Stripe</h3>

      <label class="fabric-label">New stripe width</label>
      <input class="fabric-range" id="newStripeWidth" type="range" min="5" max="250" value="55">

      <label class="fabric-label">New stripe vertical / warp color</label>
      <input class="fabric-color" id="newStripeWarp" type="color" value="#d94893">

      <label class="fabric-label">New stripe horizontal / weft color</label>
      <input class="fabric-color" id="newStripeWeft" type="color" value="#f0a4cf">

      <button class="fabric-button" onclick="addVerticalStripe()">Add Vertical Stripe</button>
      <button class="fabric-button" onclick="addHorizontalStripe()">Add Horizontal Stripe</button>

      <p class="fabric-small">After adding a stripe, drag it directly on the fabric preview.</p>

      <div id="stripeList"></div>
    </div>

    <div class="fabric-section">
      <h3>Weave / Output</h3>

      <label class="fabric-label">Output size</label>
      <select class="fabric-select" id="size">
        <option value="512">512 x 512</option>
        <option value="768" selected>768 x 768</option>
        <option value="1024">1024 x 1024</option>
        <option value="1536">1536 x 1536</option>
      </select>

      <label class="fabric-label">Vertical weave thickness / warp thickness</label>
      <input class="fabric-range" id="warpThickness" type="range" min="1" max="30" value="3">

      <label class="fabric-label">Horizontal weave thickness / weft thickness</label>
      <input class="fabric-range" id="weftThickness" type="range" min="1" max="30" value="3">
    </div>

    <div class="fabric-section">
      <h3>Loose Weave Settings</h3>

      <label class="fabric-label">Loose weave openness</label>
      <input class="fabric-range" id="looseOpenness" type="range" min="0" max="100" value="90">

      <label class="fabric-label">Loose weave irregularity</label>
      <input class="fabric-range" id="looseIrregularity" type="range" min="0" max="100" value="60">

      <label class="fabric-label">Loose thread opacity</label>
      <input class="fabric-range" id="looseThreadOpacity" type="range" min="20" max="100" value="65">

      <p class="fabric-small">
        Higher openness creates wider visible gaps. Lower thread opacity makes the cloth look airy and translucent.
      </p>
    </div>

    <div class="fabric-section">
      <h3>Waffle Weave Settings</h3>

      <label class="fabric-label">Waffle cell size</label>
      <input class="fabric-range" id="waffleCellScale" type="range" min="4" max="16" value="10">

      <label class="fabric-label">Waffle depth / raised texture</label>
      <input class="fabric-range" id="waffleDepth" type="range" min="0" max="100" value="80">
    </div>

    <div class="fabric-section">
      <h3>Centimeter Rulers</h3>

      <label class="fabric-label">
        <input id="showRulers" type="checkbox">
        Show centimeter rulers
      </label>

      <label class="fabric-label">Pixels per centimeter</label>
      <input class="fabric-range" id="pixelsPerCm" type="range" min="20" max="100" value="40">

      <p class="fabric-small">
        Small ticks = 1 mm, medium ticks = 0.5 cm, numbered ticks = 1 cm.
      </p>
    </div>

    <button class="fabric-button" onclick="renderFabricInstant()">Generate / Refresh</button>
    <button class="fabric-button secondary" onclick="downloadFabricPNG()">Download PNG</button>
  </div>

  <div class="fabric-card">
    <h2>Preview</h2>
    <p class="fabric-small">
      Use Pattern Type to switch between plain, waffle, and loose weave.
    </p>
    <canvas class="fabric-canvas" id="fabricCanvas" width="768" height="768"></canvas>
  </div>
</div>
"""

with gr.Blocks(
    title="Fabric Generator Fixed Defaults",
    theme=gr.themes.Soft(),
    head=head_html
) as demo:
    gr.Markdown("# Fabric Generator with Fixed Standard Texture Settings")
    gr.Markdown(
        "Texture amount, softness, and intersection darkness have been removed from the UI and set to standard internal defaults."
    )
    gr.HTML(body_html)

demo.launch(share=True, debug=False)

/tmp/ipykernel_3255/3032993262.py:1360: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, head. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/06/30 23:56:19 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>